# U7-L1：多传感器订阅

**课程目标**

1. 能分解一个接待任务，选择所需的感知单元和机器人基本能力，说明每项选择的用途与理由。
2. 能画出初步技术方案，说明各路信息怎样支持判断、语音与动作怎样配合，以及数据需要在什么时候更新。
3. 能通过一次读取与持续订阅的对比及多路取数实验，检查方案中的取数方式，并根据记录修改方案。

## 一、接到任务，先想需要哪些能力

校园开放日准备安排一段接待展示。组织者提出的要求是：

> 一名来宾进入接待区后，机器人能发现来宾；来宾缓慢移动时，机器人能在指定范围内调整朝向并跟进，在约定间距附近停下，再介绍展品并致意。展示结束后，操作员还要能查看机器人起点到终点的位置变化和行进中的转动读数，便于复盘表现不一致的地方。

“发现、跟进、停下、介绍、致意、复盘”分别需要什么信息和能力？前六个单元已经分别学过它们的部分用法，U4还用视觉、IMU和里程计做过联合观察。现在先回到这份具体要求，决定哪些能力值得使用，再考虑怎样调用接口。

本课先完成初步方案，并在机器人保持静止时检查方案中的取数方式。后续还需验证检测、语音与行为的交接；一张方案图和一次取数实验，尚不能证明整段接待已经实现。

### 1.1 把任务拆成可以回答的问题

先只读上面的要求，暂不查SDK方法名。把“机器人要做什么”改写为“做这一步之前要知道什么”。例如“发现来宾”首先需要知道画面里有没有人；“介绍展品”则需要准备正确文本并让机器人发声。

**活动A：两人讨论，先填前两列，再从已学能力中作选择。** 同一条信息可能支持多个步骤；一个传感器也未必能直接给出完整答案。

| 任务环节 | 需要知道什么，或输出什么 | 拟选感知单元/基本能力 | 为什么选它；它还不能回答什么 |
| --- | --- | --- | --- |
| 发现来宾 | 画面中是否有人 | 相机取图＋目标检测 | 相机给图像，检测才给候选目标；不等于确认身份 |
| 调整朝向并跟进 | | | |
| 在约定间距附近停下 | | | |
| 介绍展品并致意 | | | |
| 复盘位置变化与转动 | | | |

写完后圈出一个本组认为无需使用的能力，说明原因。例如任务没有要求听取来宾提问，是否还需要开启麦克风识别或持续对话？先按需求判断，不能因为学过某个接口就全部加入。

### 1.2 对照已学能力，检查选择理由

填完活动A后，再看下面的参考。它说明一种可以继续验证的选择，不要求所有接待任务都使用相同数量的传感器。

| 本题中的需要 | 可以选择的已学能力 | 在方案中的用途和限制 |
| --- | --- | --- |
| 来宾在哪里、画面偏向哪侧 | 相机＋person目标检测（U3、U4） | 用框的水平偏差支持方向调整；有多个候选时还需明确选择条件 |
| 是否到约定间距附近 | 本机实测的目标框高占比（U4-L2） | 在相同目标与拍摄条件下作为跟进/暂停的近似依据；不能把像素当精确米制距离 |
| 起点到终点的位置变化 | 里程计（U2、U4） | 由起终点位置估计位移；有漂移，直线位移不等于总路程，也不等于人与机器的间距 |
| 行进中转动读数怎样变化 | IMU的z轴角速度（U2） | 记录机身绕竖直轴转动的读数，辅助复盘；这一个分量不足以判断所有姿态和运动安全 |
| 说出介绍、做致意 | 有限文本语音合成与播放（U5）、已验证动作（U1、U6） | 它们属于输出和行为能力，要另外检查文本、模式、动作范围与任务退出 |

本题选择IMU和里程计，是因为要求中包含转动读数和位置变化的复盘。如果任务缩成“机器人原地发现来宾、介绍并致意”，相机、检测、语音和动作就可能构成基本方案，不必仅为凑齐三路而加入里程计。若只想看最后朝向变化，也可以讨论里程计朝向是否足够，以及IMU还会补充什么信息。

**修订一次选择表**：用不同颜色补写“必需”“为本题复盘补充”“本次不选”，每项给出对应需求。感知选择有依据之后，再进入方案组织。

### 1.3 多路信息与多模态怎样进入方案

**多传感器融合**是结合多个传感器的信息，得到对环境或机器人状态的估计或判断。例如让视觉、IMU和里程计共同支持位置估计，需要处理它们的误差、坐标和时间关系。U4让我们看到不同读数可以相互参照；如果本次只是把图像、角速度和里程计读数放在一起查看，完成的是多路联合观察，还没有得到融合定位结果。

**多模态协作**关注视觉、语音、动作等不同信息和行为方式怎样配合。在本题中，视觉提供来宾线索，语音向来宾介绍，动作表达致意。它们需要根据任务条件衔接；既要决定使用什么，也要决定什么时候运行、前一步没完成时怎样处理。

请在本组表中分别标出“感知输入”和“输出/行为”。再回答：哪一步需要多路信息共同帮助判断？哪一步需要从视觉切换到语音或动作？若暂时只做到联合观察，就写明这一层，不把未来设想写成已经实现。

## 二、为所选能力安排初步用法

### 2.1 从选择表画出一条技术路线

**活动B：画出初步方案。** 每个框写一项要完成的工作，箭头旁写“根据什么条件进入下一步”。先尝试自己画，再用下面的片段检查是否漏了判断。

```text
取图并检测来宾
    ↓ 有可用的单一目标
依据水平偏差和实测框高占比调整方向、跟进或暂停
    ↓ 停稳，并重新确认目标与现场条件
播放介绍 → 再次观察 → 条件满足才致意

行进时另记IMU转动读数、里程计位置 → 停稳后复盘
```

此图中的“另记”表示还需取得这些记录，不要求现在就创建线程；怎样让不同工作继续运行、怎样交接，后续再通过实验学习。里程计记录不替代视觉停止依据，语音播放结束也不代表来宾仍在原位。

给每个输入补一句“何时更新”。来宾可能移动，视觉判断每次都应使用新的观测；起终点位移至少需要两个时刻的位置；想了解转动过程则需要过程中的多次读数。仅在起步前保存一条数据，通常不足以满足过程观察。

| 本组要用的信息/能力 | 支持哪个判断或输出 | 何时取得或更新 | 缺失/不可用时准备怎样处理 |
| --- | --- | --- | --- |
| 来宾视觉线索 | | | |
| IMU、里程计（按选择填写） | | | |
| 介绍与致意 | | | |

这是一份可修改的初步方案。暂时不知道的条件标为“待验证”，后面的实验用来回答其中的取数问题；本课不要求写完整项目的模块分工、联调和展演计划。

### 2.2 这些信息一次取就够，还是要持续更新

以IMU为例，`imu = robot.get_imu()`主动取得一次最近可用的数据。若之后还想查看，就再调用一次。把主动读取放在循环中，也可以得到多次读数，称为**轮询**；更新节奏受循环中其他工作和等待影响。

另一种方式是先定义`on_imu(imu)`，再把函数交给`robot.subscribe_imu(on_imu)`登记。SDK收到订阅更新时调用这个函数，并把该条数据传入，这就是**回调**。函数名后不加括号，表示把函数交给SDK，而非立即自己调用它。订阅也可能有排队和丢弃，不保证每条原始采样都能处理。

| 用数需要 | 可以先考虑什么 | 还要验证什么 |
| --- | --- | --- |
| 只看启动前当前模式或一份读数 | 一次主动读取 | 返回内容是否足够，后面是否还会用旧值 |
| 在自己安排的几个时刻检查 | 循环中主动读取 | 一轮多久，其他工作是否推迟了下一次读取 |
| 数据到达时持续保存最近记录 | 订阅回调 | 收到什么、回调做多长时间、什么时候退订 |

订阅是候选方式之一。先运行下面对比，观察实际差别，再决定它是否适合本组过程记录；不能把“需要多次读数”直接等同于“只能用订阅”。

⚠️ **实验安全**：以下只做静止取数，不执行方案中的跟进、语音或动作，不发运动或模式切换指令。调整站姿时先停止采集，按既定人工保护流程操作。

📌 **前置**：其他Notebook的旧订阅和活动任务已退出，在Notebook所在目录运行。

**STEP 01：运行连接格，核对本组机器人与当前模式。** 连接异常先处理，不继续创建订阅。

In [2]:
import time
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()
print("机器人：", robot.robot_info.model, robot.robot_info.serial_number)
print("模式：", robot.get_mode())

[Booster OS] [boosteros.sdk] [INFO] Starting Booster SDK Local Adapter (mode=local, network_interface=default, virtual_robot_name=default).
[Booster OS] [boosteros.sdk] [INFO] Booster SDK Local Adapter started (mode=local, network_interface=default).


ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024


[Booster OS] [boosteros.robot] [INFO] Fetching camera configuration from API...
[Booster OS] [boosteros.robot] [INFO] Using cameras ['head'] from API, default camera is 'head'.
[Booster OS] [boosteros.BoosterHandGuidingManager] [INFO] BoosterHandGuidingManager initialized with default_record_file=/opt/booster/Gait/bin/joint_position.txt, record_archive_dir=~/.boosteros/hand_guiding
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_43895d9f] Booster ROS 2 Adapter initialized. Cameras: head(rgb=/boostercamera/head/rgb, depth=/boostercamera/head/depth). Default camera: head.
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_43895d9f] Booster ROS 2 Adapter started with 4 threads.
[Booster OS] [boosteros.robot] [INFO] Initializing audio manager & speech runtime...
[Booster OS] [boosteros.BoosterAudioManager] [INFO] _BoosterAudioRuntime initialized
[Booster OS] [boosteros.BoosterAudioManager] [INFO] BoosterAudioManager initialized


ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024

[Booster OS] [boosteros.BoosterSpeechRuntime] [INFO] Booster speech runtime initialized for robot_name=''
[Booster OS] [boosteros.robot] [INFO] Booster OS resources initialized.


机器人： Booster K1 01022026030137
模式： prepare


**STEP 01A：先主动读取一次，过两秒再看保存的值。** 下一格把这次的时间戳和角速度提取到普通变量中，等待后再次打印。保存下来的这两个数不会自行更新；最后再调用一次`get_imu()`，取得另一份读数作为对照。

机器人保持静止，角速度可能很接近，不能只看数值是否变化；也要查看时间戳。源数据未更新时，两次主动读取的时间戳可能相同，如实记录，不为制造差异去推机器人。这两秒只是观察间隔，不代表SDK延迟。

In [ ]:
once_imu = robot.get_imu()
once_stamp = once_imu.timestamp
once_wz = once_imu.angular_velocity[2]
print("第一次取得并保存：", once_stamp, "z轴角速度：", once_wz)
time.sleep(2.0)
print("两秒后，原来保存的数仍是：", once_stamp, "z轴角速度：", once_wz)
again_imu = robot.get_imu()
print("现在重新调用get_imu：", again_imu.timestamp,
      "z轴角速度：", again_imu.angular_velocity[2])

**STEP 01B：登记订阅，让回调保存收到的更新。** 下一格中`on_compare_imu`每被调用一次就把次数加1，并保存这一条的时间戳与角速度；外面的循环每秒打印一次最近记录，共三次。循环没有再次调用`get_imu()`，新记录来自SDK对回调的调用。

`comparison["IMU"]`每次替换为一份新记录，`current`让这一轮打印使用同一份记录。`with 订阅对象:`下面的缩进是本次订阅使用范围，离开时退出订阅。等待整格结束，记录回调次数和最近时间戳；没有更新时先核对数据与连接，不自动判断为方式失效。次数只表示本实验的回调处理次数，不是传感器全部采样数。退出后的变量仍保留最后记录，但不会因本次订阅继续更新。

| 观察项目 | 一次主动读取 | 持续订阅 |
| --- | --- | --- |
| 再次显示之前，是谁取得了新数据 | | |
| 原变量/最近记录有没有改变，依据是什么 | | |
| 若要看行进过程，还应怎样安排 | | |
| 何时结束本次取数 | | |

In [ ]:
comparison = {"IMU": {"count": 0, "stamp": None, "wz": None}}

def on_compare_imu(imu):
    previous = comparison["IMU"]
    comparison["IMU"] = {"count": previous["count"] + 1,
                         "stamp": imu.timestamp, "wz": imu.angular_velocity[2]}

with robot.subscribe_imu(on_compare_imu, queue_size=1, overflow="drop_oldest"):
    for number in range(3):
        time.sleep(1.0)
        current = comparison["IMU"]
        print("观察", number + 1, "回调次数：", current["count"],
              "最近时间戳：", current["stamp"], "z轴角速度：", current["wz"])
print("本次IMU对比订阅已退出")

## 三、检验方案中的多路取数

### 3.1 选定字段，保存各路最近记录

回到活动B：本题用视觉支持来宾判断，用里程计位置与IMU转动读数作复盘。现在先检查这几条数据通道能否持续取得。公共样例选择三路订阅，让数据到达时保存最近记录，再每秒显示一次；它是本题一种取数安排，可与主动轮询方案比较。

这里仅打印原始数据：图像尺寸和中心像素用于确认画面变化，尚没有调用person检测；静止的IMU/里程计只用于核查取数，不构成一次行进轨迹记录。方案中的视觉判断、跟进和致意仍需各自验证。

`latest`是字典，三个键分别对应图像、IMU、里程计，开始时为`None`，表示尚无记录。回调只提取数值、记时间、保存记录，不在每次到达时打印整表，这种仅做必要工作的写法称为短回调。

| 字段 | 具体含义 | 例子 |
| --- | --- | --- |
| `value` | 选取的传感数值 | IMU的z轴角速度 |
| `source` | SDK数据对象附带的源时间戳 | `imu.timestamp`原值 |
| `received` | 回调开始处理这条数据时记下的本地时刻 | `time.monotonic()`返回值 |

每次回调把值和时间装进同一份新记录，再替换该路，避免把新数值与旧时间拼在一起。`snapshot = dict(latest)`取得各路当时记录的引用，供这一轮打印；它并不让三个传感器在同一时刻采样。

`time.monotonic()`用于本程序内算经过时长。例如100.0秒进入回调，100.2秒打印，则“回调记录后”经过0.2秒；这不包含进入回调前可能发生的传输或排队，也不能直接与源时间戳相减。

**STEP 02：运行下一格，等待五组输出全部结束。** 准备一张普通彩色纸，仅在画面中心缓慢移入移出，观察中心像素，不碰机器人。每组抄下三路数值与时间；出现“尚未收到”时保留缺失提示，不补0。

三个订阅在同一个`with`中管理，行末反斜杠只是分行书写；正常或Python异常退出该范围时执行清理。若连接中断或内核被强制结束，仍需核查实际资源。不要在上一格未结束时重开订阅。

In [3]:
latest = {"图像": None, "IMU": None, "里程计": None}

def on_image(image):
    received = time.monotonic()
    pixels = image.to_numpy()
    latest["图像"] = {"source": image.header.timestamp, "received": received,
                      "value": f"{image.width}×{image.height}，中心像素{pixels[image.height//2, image.width//2].tolist()}"}

def on_imu(imu):
    latest["IMU"] = {"source": imu.timestamp, "received": time.monotonic(),
                     "value": f"z轴角速度={imu.angular_velocity[2]:.3f} rad/s"}

def on_odom(odom):
    x, y, yaw = odom.pose_2d
    latest["里程计"] = {"source": odom.timestamp, "received": time.monotonic(),
                        "value": f"x={x:.3f} m，y={y:.3f} m，yaw={yaw:.3f} rad"}

with robot.subscribe_image(on_image, queue_size=1, overflow="drop_oldest"), \
     robot.subscribe_imu(on_imu, queue_size=1, overflow="drop_oldest"), \
     robot.subscribe_odom(on_odom, queue_size=1, overflow="drop_oldest"):
    for number in range(5):
        time.sleep(1.0)
        snapshot = dict(latest)
        observed = time.monotonic()
        print(f"\n观察{number+1}，本地时刻={observed:.3f}")
        for name in ["图像", "IMU", "里程计"]:
            record = snapshot[name]
            if record is None:
                print(name, "尚未收到")
            else:
                age = observed-record["received"]
                print(name, record["value"], "源时间戳=", record["source"],
                      f"回调记录后{age:.3f}秒")
print("三路观察已结束，订阅已退出")


观察1，本地时刻=13479.995
图像 544×448，中心像素[135, 131, 134] 源时间戳= 1793268290.4957242 接收后0.028秒
IMU z轴角速度=-0.001 rad/s 源时间戳= 1793268290.5454278 接收后0.000秒
里程计 尚未收到

观察2，本地时刻=13480.996
图像 544×448，中心像素[135, 131, 134] 源时间戳= 1793268291.4623897 接收后0.037秒
IMU z轴角速度=-0.001 rad/s 源时间戳= 1793268291.6513515 接收后0.010秒
里程计 尚未收到

观察3，本地时刻=13482.011
图像 544×448，中心像素[135, 131, 134] 源时间戳= 1793268292.4957445 接收后0.007秒
IMU z轴角速度=0.000 rad/s 源时间戳= 1793268292.5969625 接收后0.002秒
里程计 尚未收到

观察4，本地时刻=13483.012
图像 544×448，中心像素[135, 131, 134] 源时间戳= 1793268293.495722 接收后0.035秒
IMU z轴角速度=0.002 rad/s 源时间戳= 1793268293.6147451 接收后0.050秒
里程计 尚未收到

观察5，本地时刻=13484.016
图像 544×448，中心像素[135, 131, 134] 源时间戳= 1793268294.529058 接收后0.011秒
IMU z轴角速度=-0.001 rad/s 源时间戳= 1793268294.564373 接收后0.010秒
里程计 x=-0.011 m，y=-0.003 m，yaw=-1.372 rad 源时间戳= 1793268294.590429 接收后0.001秒
三路观察已结束，订阅已退出


先抄至少两轮完整输出。机器人静止时，新数值可能和旧值很接近，是否更新还要看时间记录。填写“这路数据用在哪里”，检查取数与前面选择是否对应。

| 数据路 | 两轮实际值及源时间戳 | 回调记录后时长 | 支持方案的哪项需要 | 缺失或待查项 |
| --- | --- | --- | --- | --- |
| 图像 | | | | |
| IMU | | | | |
| 里程计 | | | | |

### 3.2 看到最近记录，还应检查什么

三路都刚进入回调，不代表对应同一采集时刻；相机画面可能先排队，再送进回调。因此源时间戳要保留，先看同一路是否更新。资料不足时不能仅凭数值相近就比较跨路采集先后。

**STEP 03：先打印最后一轮的源时间戳原值，写下已核实与待核项。** 原值未变化时继续查数据来源，不立刻判断传感器损坏。

| 要核实什么 | 怎样核实 | 本组依据/待核项 |
| --- | --- | --- |
| 同一路有没有更新 | 对照STEP 02两轮原值及本地接收记录 | |
| 源字段单位 | 查本机SDK版本的图像、IMU、Odom时间字段，由教师补充资料 | |
| 各路计时基准是否相同 | 查时钟来源说明，必要时由教师向技术人员确认；数值相近不能证明 | |

**选做：跨路源时刻比较。** 只有单位均为秒、计时基准相同都有明确依据时，再把下一格`TIMESTAMPS_VERIFIED`改为`True`计算差值。否则保持`False`，本课仍可凭原值与本地记录完成取数检查。差值只描述这几份记录的源时刻关系，不表示完整延迟，也不代表已做同步配对或融合定位。

In [4]:
for name in ["图像", "IMU", "里程计"]:
    record = snapshot[name]
    print(name, "源时间戳原值：", None if record is None else record["source"])

TIMESTAMPS_VERIFIED = False  # 有字段单位与共同计时基准的依据后，才由教师指导修改
if not TIMESTAMPS_VERIFIED:
    print("已保存原值；跨路差值暂不换算，请填写核对依据或待核项")
elif any(snapshot[name] is None for name in ["图像", "IMU", "里程计"]):
    print("缺少记录，不能比较")
else:
    print("图像－IMU：", (snapshot["图像"]["source"]-snapshot["IMU"]["source"])*1000, "毫秒")
    print("图像－里程计：", (snapshot["图像"]["source"]-snapshot["里程计"]["source"])*1000, "毫秒")

本次仅保留源时间戳原值，单位与基准待核


## 四、拓展：数据来得比处理快时怎么办

**有余力或课后再做本节；课堂先完成第五部分的方案修订。** 本节帮助进一步解释订阅参数，不以完成慢回调实验作为本课方案设计的必达条件。

若在图像回调里做较慢检测或写文件，上一张还没处理完，下一张可能到了。例如每0.1秒到一张图，处理却要0.3秒，等待的记录会增加。订阅用**队列**暂存等待处理的数据，`queue_size`限制暂存数量，`overflow`决定满了如何处理；容量0表示不设上限，不等于不排队。

| 队列满时的策略 | 发生什么 | 取舍 |
| --- | --- | --- |
| `drop_oldest` | 丢最旧待处理项，给新记录留位 | 偏向近期状态，会漏中间记录 |
| `drop_newest` | 丢新到的，保留已排队项 | 保留等待顺序，处理到的记录可能较旧 |
| `block` | 等空位 | 会增加等待，不能保证整条采集链路无损 |

例如A处理中、B在唯一等待位、C新到：前一种留下C，后一种留下B，都不会撤回正在处理的A。STEP 02先用容量1与`drop_oldest`作近期状态观察，但该选择仍有遗漏数据的代价。

### 4.1 用相同处理条件比较两种策略

**STEP 04：先定义下面函数，尚不启动订阅。** 它特意在回调中等0.3秒，用于暴露排队；每次观察约三秒，退出后打印。实际是否积压以本机结果为准。两次只改策略，其他条件相近；不通过彩色纸推算丢帧，处理条数也不代表全部到达消息。

In [5]:
def queue_trial(policy):
    rows = []
    def slow_image(image):
        received = time.monotonic()
        time.sleep(0.3)
        rows.append((image.header.timestamp, received))
    with robot.subscribe_image(slow_image, queue_size=1, overflow=policy):
        time.sleep(3.0)
    print("策略：", policy, "已处理条数：", len(rows))
    for stamp, received in rows[-5:]:
        print("源时间戳：", stamp, "回调开始的本地时刻：", round(received, 3))
    return rows

**STEP 05：运行`drop_oldest`这一格，等待函数全部返回。** 记下已处理条数，以及末几条源时间戳和回调开始的本地时刻。本机图像如果更新得足够快，队列满时会优先留下新到的数据；本次是否观察到这种现象，要结合下一次记录判断。

In [6]:
oldest_rows = queue_trial("drop_oldest")

策略： drop_oldest 已处理条数： 10
源时间戳： 1793268669.730073 回调开始的本地时刻： 13859.256
源时间戳： 1793268670.0300758 回调开始的本地时刻： 13859.558
源时间戳： 1793268670.3633997 回调开始的本地时刻： 13859.87
源时间戳： 1793268670.663406 回调开始的本地时刻： 13860.172
源时间戳： 1793268670.9634206 回调开始的本地时刻： 13860.473


**STEP 06：在相同条件下运行`drop_newest`。** 本次保留已排队的记录；与STEP 05并列抄下末几条记录，比较源时间戳的变化和回调开始间隔。两个实验在不同时段运行，不能直接把绝对时间戳大小当成策略优劣；关注各自记录内部的变化。

同样的0.3秒等待会使两次处理条数接近，不能因此认定接到的是同样新旧的数据。只有核实源单位和计时基准后，才进一步换算源间隔；即使换算成立，本实验也不足以精确量出排队延迟。差异不明显时如实记录，并用“A处理中、B等待、C到达”的例子解释两种策略的取舍。

In [7]:
newest_rows = queue_trial("drop_newest")

策略： drop_newest 已处理条数： 10
源时间戳： 1793268704.5301192 回调开始的本地时刻： 13894.317
源时间戳： 1793268704.8967917 回调开始的本地时刻： 13894.624
源时间戳： 1793268705.1301312 回调开始的本地时刻： 13894.936
源时间戳： 1793268705.43012 回调开始的本地时刻： 13895.238
源时间戳： 1793268705.7634563 回调开始的本地时刻： 13895.551


### 4.2 根据任务说明选择与代价

| 策略 | 已处理条数 | 末几条记录的变化 | 支持什么判断 | 仍不能确认什么 |
| --- | --- | --- | --- | --- |
| `drop_oldest` | | | | |
| `drop_newest` | | | | |

想看近期状态，可用小队列并偏向新记录；若要保存完整过程供分析，漏中间数据可能不可接受，还需重新考虑处理能力和存储。三路订阅不会自动同步，也不会加快正在处理的一条数据。差异不明显时如实记录，并用A/B/C例子说明取舍，不猜测准确丢帧率。

## 五、把实验结果写回初步方案

**活动C：回到活动A、B，修改原来的选择与取数安排。** 用本次输出说明哪些部分有了依据，哪些仍只是设想。至少写出一处“保留或修改的决定＋依据”，不要只写“全部正常”。

| 方案问题 | 本组决定 | 实验或需求依据 | 还需验证什么 |
| --- | --- | --- | --- |
| 本题选哪些感知与行为能力；缩为原地接待后可去掉什么 | | | |
| 哪些值只看一次，哪些需过程更新；何处选轮询或订阅 | | | |
| 保存哪些字段，怎样标明尚未收到或记录未更新 | | | |
| 播报后如何取得新观测，行为满足什么条件才开始 | | | |

例如“选IMU订阅保存转动读数”可由过程复盘需求及STEP 01B的回调记录支持；但本课静止试验尚未记录实际行进过程，完整轨迹存储也还没设计。若选择主动轮询，应写明放在什么循环、何时再次读取，并把取图或播报会不会推迟下一次读取列为待测。

本课交一张修订后的能力选择表和初步方案图，并附一次读取/订阅对比、多路取数记录与待核项。后续L2在足球问题中进一步学习球坐标和自动踢球，L3研究播报期间继续检测，L4验证局部交接；完整项目的需求验收、模块分工、整合与彩排留到U8。

### 5.1 学习检查

1. 指出本题一项需求，说明为什么选某个感知单元；再说出一个本次不选的能力及理由。
2. 介绍和致意分别是什么能力？与相机输入怎样配合？本课三路打印是否已经完成融合定位，依据是什么？
3. STEP 01A过两秒后原来保存的数为什么不变？主动读取放进循环能否持续取数？与订阅相比需要自行安排什么？
4. STEP 02的实际记录证明了方案哪一部分可行？哪些目标检测、行为和时序条件还没验证？

### 5.2 完成与收尾

基本完成：能力选择有需求依据，方案写出信息如何使用和更新，至少一处决定能联系实验记录，未验证项明确。跨路时间换算与慢队列实验按条件选做。

确认STEP 01B、STEP 02及选做格的订阅均已退出；异常中断先核查旧订阅，不重启内核代替收尾。机器人保持安全姿态，需要关机时按既定人工流程操作。